In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from satchip import models, download_data, merge_modality, generate_labels

/home/wbhorn/miniforge3/envs/satchip/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
modality = models.HLS_S30

DATA_PATH = Path.cwd() / 'data' 
MODALITY_PATH = DATA_PATH / modality['id']

RAW_DATA_PATH = MODALITY_PATH / 'raw'
MERGED_PATH = MODALITY_PATH / 'merged'
REPROJECTED_PATH = MODALITY_PATH / 'wgs84'
STACKED_PATH = MODALITY_PATH / 'stacked'
WARPED_PATH = MODALITY_PATH / 'warped' 

In [3]:
shp_path = "hwds_pristine"
df = gpd.read_file(shp_path)

df["SwathDate"] = pd.to_datetime(df["SwathDate"], format="%Y-%m-%d")
df["HLSDate"] = pd.to_datetime(df["HLSDate"], format="%Y-%m-%d")

In [4]:
swath = df.iloc[1]
swath

SwathDate                                   2019-07-09 00:00:00
HLSID                                                      102a
MODISID                                                     102
Visibility                                                    2
HLSDate                                     2019-07-17 00:00:00
geometry      POLYGON Z ((-99.49996223199997 40.270482574000...
Name: 1, dtype: object

In [ ]:
event = models.Event(
    name=swath['HLSID'],
    date=swath['SwathDate'],
    wgs84_geometry=swath['geometry'],
    buffer_m=10000
)
print(event)

Event(name='102a', date=Timestamp('2019-07-09 00:00:00'), wgs84_geometry=<POLYGON Z ((-99.5 40.27 0, -99.476 40.268 0, -99.458 40.275 0, -99.428 40.2...>)


In [6]:
local_files = download_data.download_data(event, modality, RAW_DATA_PATH)

Logging in to earthaccess


QUEUEING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 10439.36it/s]
PROCESSING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 191617.95it/s]
COLLECTING RESULTS | : 100%|██████████| 18/18 [00:00<00:00, 424143.10it/s]


In [7]:
merged_event = merge_modality.merge_modality(local_files, modality, event=event, output_path=MERGED_PATH)

In [8]:
merged_event

[PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.B.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.G.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.R.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.N.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.SW1.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.SW2.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.Fmask.tif')]

In [9]:
stacked_filename = MERGED_PATH / f'{swath["HLSID"]}.stacked.tif'
data_bands, fmask = merged_event[:-1], merged_event[-1]  

stacked = merge_modality.stack_bands(data_bands, stacked_filename)

In [10]:
data_bands

[PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.B.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.G.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.R.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.N.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.SW1.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged/102a.HLS_S30.2019-07-09.SW2.tif')]

In [11]:
stacked, fmask = merge_modality.reproject_files([stacked, fmask], REPROJECTED_PATH)

In [12]:
label = generate_labels.binary_mask_from_template(stacked, event, REPROJECTED_PATH)

generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84/102a.MASK.tif


In [13]:
warped = merge_modality.warp_to_reference(
    reference_path=label, 
    data_files=[stacked, fmask], 
    output_dir=WARPED_PATH, 
    bounding_box_4326=event.wgs84_geometry.bounds
)